# 03 — Single Legs & Stock Lab: Build, Summarize, Compare

You will build all six structures with `strategies.*`, read each one's risk with
`analyzer.summarize`, draw payoffs with `viz.plot_payoff`, and put a covered call head-to-head
with plain stock.

DEMO: spot **$100**, IV **0.25**, **45 DTE**. Chain mids from module 00.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, payoff, viz
from optionslab.position import StockLeg

SPOT, VOL, t = 100.0, 0.25, 45/365

## Helper: pretty-print a summary

`analyzer.summarize` returns a dict (net premium, breakevens, max P/L, POP, expected move,
greeks, DTE). We format the key fields.

In [ ]:
def show(pos):
    s = analyzer.summarize(pos, SPOT, VOL)
    print(s['label'])
    print(f"  net_premium {s['net_premium']:+.0f}  (+debit/-credit)")
    print(f"  breakevens  {[round(b,2) for b in s['breakevens']]}")
    print(f"  max_profit {s['max_profit']:.0f}   max_loss {s['max_loss']:.0f}")
    print(f"  POP {s['probability_of_profit']:.2f}   delta {s['greeks'].delta:+.1f}")

## 1. Long call and long put (debit, long-vega, short-theta)

In [ ]:
lc = strategies.long_call((100, 3.91), expiry=t)
lp = strategies.long_put((100, 3.42), expiry=t)
show(lc); print(); show(lp)

Note the long call's breakeven at **103.91** (strike + premium) and capped max loss = the debit.
Both are POP < 0.5 — the stock must *move*, and move enough to clear the premium.

## 2. Covered call and cash-secured put (credit, short-vega, long-theta)

In [ ]:
cc  = strategies.covered_call(100, (105, 1.85), expiry=t)
csp = strategies.cash_secured_put((95, 1.58), expiry=t)
show(cc); print(); show(csp)

The covered call caps max profit at **685** (5 points of upside + 1.85 credit) and the CSP keeps
the **158** credit if DEMO holds above 95. Both show positive theta and high POP — the seller's
profile.

## 3. Protective put and collar (hedged stock)

In [ ]:
pp = strategies.protective_put(100, (95, 1.58), expiry=t)
col = strategies.collar(100, (95, 1.58), (105, 1.85), expiry=t)
show(pp); print(); show(col)

The protective put floors max loss at **-658**; the collar boxes the outcome to about
**-473 / +527** for a small net credit — the sold call pays for the bought put.

## 4. Payoff diagram: the long call

`viz.plot_payoff` draws the expiry line; passing `vol` adds the mark-to-model 'now' curve and
`spot` marks the current price.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
viz.plot_payoff(lc, spot=SPOT, vol=VOL, ax=ax)
ax.set_title('Long 100 call — capped loss, open-ended upside')
plt.show()

## 5. Covered call vs. plain stock

Build plain long stock with `strategies.custom` + a `StockLeg`, then overlay both expiry payoffs
with `viz.plot_compare`. See exactly where the credit helps and where the cap hurts.

In [ ]:
stock = strategies.custom(StockLeg(100, SPOT), label='Long 100 shares')
fig, ax = plt.subplots(figsize=(8, 4.5))
viz.plot_compare([stock, cc], ax=ax)
ax.set_title('Covered call vs plain stock (P&L at expiry)')
plt.show()

Below ~105 the covered call sits **above** stock by the premium cushion; above 105 stock keeps
climbing while the covered call flatlines at its cap. You sold the upside tail for a sure credit.

## 6. Where do they cross? (quantify the trade-off)

Use `payoff.pnl_curve` to compute both P&Ls on a grid and find the crossover and the downside
cushion numerically.

In [ ]:
grid = np.linspace(85, 120, 71)
pnl_stock = payoff.pnl_curve(stock, grid)
pnl_cc    = payoff.pnl_curve(cc, grid)
cushion = grid[np.argmin(np.abs(pnl_cc))]      # cc breakeven ~98.15
cross   = grid[np.argmin(np.abs(pnl_cc - pnl_stock))]
print(f'covered-call breakeven near spot {cushion:.1f}')
print(f'covered call and stock give equal P&L near spot {cross:.1f} (the short strike)')

## Experiments

1. In section 2, sell a **107.5** call for the covered call instead of 105 (mid ~1.19). More
   upside kept, less premium — how do max_profit and the cushion change?
2. In section 1, buy the **105** call (~1.85) instead of ATM. Compare breakeven and POP — what
   does the cheaper, further-OTM strike cost you in probability?
3. In section 3, widen the collar to a **90 put / 110 call**. Is it still a credit? How does the
   max-loss/max-profit band change?
4. Run `analyzer.summarize` on the covered call at `vol=0.40` instead of 0.25. Which fields move,
   and why does a covered-call seller *prefer* higher entry IV?
5. Build a cash-secured put at the **90** strike (~0.62). Compare its POP and max loss to the 95
   version — the classic 'further OTM = higher POP, less credit' trade-off.